In [ ]:
from builda_fmu import get_configured_builda_fmu
from buildyn.walker.interval_walker import IntervalWalker
from buildyn.walker.random.ramp_walker import RampWalker
from buildyn.walker.fixed.poisson_walker import PoissonWalker
from buildyn.walker.random.sinusoidal_walker import SinusoidalWalker

fmu = get_configured_builda_fmu(internal_controller=False)
fmu.set_initial_variables({})
observables = ["thermalZone.TAir", "ctrSignalHeating", "weaBus.TDryBul"]

ramp_walker = RampWalker(min=0, max=1)
pois_walker = PoissonWalker(min=0, max=1, lam=8)
sin_walker = SinusoidalWalker(min=0, max=1)
interval_walker = IntervalWalker(walker=sin_walker, interval=900)
walkers = {
    "ctrSignalHeating": interval_walker
}

df = fmu.simulate(observables=observables, walker=walkers)

df[["thermalZone.TAir", "ctrSignalHeating"]].plot(secondary_y="ctrSignalHeating")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker

observables_top = ["ctrSignalHeating", "thermalZone.TAir"]
observables_bottom = ["weaBus.TDryBul", "weaBus.HDirNor"]

# consistent colors per observable
color_map = {
    "ctrSignalHeating": "tab:blue",
    "thermalZone.TAir": "tab:red",
    "weaBus.TDryBul": "tab:green",
    "weaBus.HDirNor": "tab:purple"
}

name_map = {
    "ctrSignalHeating": "Heating Signal",
    "thermalZone.TAir": "Indoor Temperature [°C]",
    "weaBus.TDryBul": "Outdoor Temperature [°C]",
    "weaBus.HDirNor": "Direct Irradiance [W/m²]"
}

walkers_dict = {
    "Ramp": RampWalker(min=0, max=1),
    "Poisson": PoissonWalker(min=0, max=1, lam=8),
    "Sinusoidal": SinusoidalWalker(min=0, max=1),
}

scenario_names = ["Winter", "Spring", "Summer", "Autumn"]

n_walkers = len(walkers_dict)
n_scenarios = 4

step = 900
day_len = 96 * step

start_times = [
    0,
    90 * 24 * 3600,
    180 * 24 * 3600,
    270 * 24 * 3600
]

fig = plt.figure(figsize=(6 * n_scenarios, 10))

outer = gridspec.GridSpec(
    2,
    n_scenarios,
    height_ratios=[3, 1],
    wspace=0.3,
    hspace=0.15
)

for scen in range(n_scenarios):

    print(f"Scenario {scen+1} ({scenario_names[scen]}) start={start_times[scen]}")

    start_time = start_times[scen]
    stop_time = start_time + day_len

    # -------------------------
    # TOP BLOCK (walkers × 2)
    # -------------------------
    top_grid = gridspec.GridSpecFromSubplotSpec(
        n_walkers,
        len(observables_top),
        subplot_spec=outer[0, scen],
        wspace=0.15,
        hspace=0.25
    )

    last_df = None

    for w_idx, (walker_name, walker_obj) in enumerate(walkers_dict.items()):

        fmu = get_configured_builda_fmu(internal_controller=False)
        fmu.set_initial_variables({})

        interval_walker = IntervalWalker(
            walker=walker_obj,
            interval=900
        )

        walkers = {"ctrSignalHeating": interval_walker}

        df = fmu.simulate(
            observables=observables_top + observables_bottom,
            walker=walkers
        )
        
        # Claculate Kelvin in °C
        df["thermalZone.TAir"] = df["thermalZone.TAir"] - 273.15
        df["weaBus.TDryBul"] = df["weaBus.TDryBul"] - 273.15

        df = df.iloc[1:]

        last_df = df

        for o_idx, obs in enumerate(observables_top):

            ax = fig.add_subplot(top_grid[w_idx, o_idx])
            
            if obs == "thermalZone.TAir":
                ax.yaxis.set_label_position("right")
                ax.yaxis.tick_right()
                
            
            import numpy as np
            x_hours = np.arange(len(df)) * (900 / 3600)
            ax.plot(x_hours, df[obs], color=color_map.get(obs))
            
            ax.set_xlim(0, 24)

            ax.xaxis.set_major_locator(mticker.MultipleLocator(6))
            ax.xaxis.set_minor_locator(mticker.MultipleLocator(1))

            ax.grid(True, which="major", linestyle="-", alpha=0.4)
            ax.grid(True, which="minor", linestyle=":", alpha=0.2)

            ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%d'))
            #ax.set_xlabel("Time [h]")

            if w_idx == 0:
                ax.set_title(name_map.get(obs, obs), fontsize=10)

            # ✔ walker only once per row (left side only)
            if o_idx == 0 and scen == 0:
                ax.set_ylabel(
                    walker_name,
                    fontsize=12,
                    fontweight="bold"
                )
            else:
                ax.set_ylabel("")

    # -------------------------
    # BOTTOM BLOCK (1 × 2)
    # -------------------------
    bottom_grid = gridspec.GridSpecFromSubplotSpec(
        1,
        len(observables_bottom),
        subplot_spec=outer[1, scen],
        wspace=0.25
    )

    for b_idx, obs in enumerate(observables_bottom):

        ax = fig.add_subplot(bottom_grid[0, b_idx])
        
        if obs == "weaBus.HDirNor":
                ax.yaxis.set_label_position("right")
                ax.yaxis.tick_right()
        
        x_hours = np.arange(len(last_df)) * (900 / 3600)
        ax.plot(x_hours, last_df[obs], color=color_map.get(obs))

        ax.set_xlim(0, 24)

        ax.xaxis.set_major_locator(mticker.MultipleLocator(6))
        ax.xaxis.set_minor_locator(mticker.MultipleLocator(1))

        ax.grid(True, which="major", linestyle="-", alpha=0.4)
        ax.grid(True, which="minor", linestyle=":", alpha=0.2)

        ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%d'))
        #ax.set_xlabel("Time [h]")

        ax.set_title(name_map.get(obs, obs), fontsize=10)

    # -------------------------
    # SCENARIO TITLE (above column)
    # -------------------------
    ax_title = fig.add_subplot(outer[:, scen])
    ax_title.set_title(scenario_names[scen], fontsize=14, fontweight="bold", pad=20)
    ax_title.set_frame_on(False)
    ax_title.set_xticks([])
    ax_title.set_yticks([])

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig("exciting_scenarios.pdf", dpi=300)